In [25]:
# Generate MISMIP geometry
# Author: Neosha Narayanan, date: April 2026
# Based on generate_idealized_geometry.ipynb from June 2025

In [26]:
%matplotlib qt

In [27]:
# Based on Gudmundsson (2012) https://tc.copernicus.org/articles/6/1497/2012/tc-6-1497-2012.pdf 
# which informed the MISMIP simulations shown in https://gmd.copernicus.org/articles/9/2471/2016/gmd-9-2471-2016.pdf

In [3]:
import sys; print(sys.executable)

/opt/homebrew/Cellar/jupyterlab/4.5.6_2/libexec/bin/python


In [1]:
import numpy as np

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

## Bed

In [4]:
xbar = 12000 # Characteristic length scale of the glacier (along flow length)

B0 = 5000 # bedrock topography at x = 0
B2 = -640
B4 = 288
B6 = -48
dc = 1000 # controls depth of channel
fc = 400 # slope of the flanks
omega_c = 400 # m (50 km) half-width of the channel
Ly = 500 # domain width (y-dir)
Lx = 12000


In [28]:
x = np.linspace(0, Lx, 1000)
y = np.linspace(0, Ly, 100)

X, Y = np.meshgrid(x, y)
xchar = 7500
Bx = 5000 + B2*(X/xchar)**2 + B4 *(X/xchar)**4 + B6*(X/xchar)**6
By = (dc/(1+np.exp(-2*(Y- Ly/2-omega_c)/fc))) + (dc/(1+np.exp(2*(Y - Ly/2 + omega_c)/fc)))

B = Bx + By

In [29]:
# Add bed roughness: 



In [30]:
fig = plt.figure(figsize=(12, 6))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(X, Y, B, cmap='copper', alpha=0.8, edgecolor='none', label='bed1')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('z')
ax.set_box_aspect(None, zoom=0.85)
plt.axis('scaled')
#ax.set_aspect('equal')

(np.float64(-875.0),
 np.float64(12875.0),
 np.float64(-36.45833333333333),
 np.float64(536.4583333333334),
 np.float64(4668.065707898794),
 np.float64(5372.231504092613))

## Surface

In [31]:
# We want the surface to be mostly flat and for the thickness to decrease towards the terminus
# Use the same X, Y meshgrid

tmax = 200 # maximum ice thickness at edges
tmin = 50 # minimum ice thickness at edges

Sx = B[0, :]
Sx = Sx + (tmax-tmin)/Lx # !! Need to change this so that the glacier thickness decreases downstream
Sy = 0*Y
S = Sx + Sy

In [32]:
fig = plt.figure(figsize=(12, 6))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(X, Y, S, cmap='Blues', alpha=0.8, edgecolor='none', label='surf')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('z')
ax.set_box_aspect(None, zoom=0.85)
plt.axis('scaled')


(np.float64(-875.0),
 np.float64(12875.0),
 np.float64(-36.45833333333333),
 np.float64(536.4583333333334),
 np.float64(4790.302170835403),
 np.float64(5369.74963750207))

In [33]:
# Plot surface and bed together
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(X, Y, S, cmap = 'Blues', alpha=0.8, edgecolor='none', label='surf')
ax.plot_surface(X, Y, B, cmap='copper', alpha = 0.8, edgecolor='none', label='bed')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('z')
ax.set_box_aspect(None, zoom=0.85)
#plt.axis('scaled')


### export to .mat file

In [34]:
import scipy.io

In [35]:
x_collapsed = X[0, :]
y_collapsed = Y[:, 0]

In [242]:
surface_dict = {'X':X, 'Y': Y,'surface':S}
bed_dict = {'X':X, 'Y':Y, 'bed':B}

In [243]:
surface_dict = {'X':x_collapsed, 'Y': y_collapsed,'surface':S}
bed_dict = {'X':x_collapsed, 'Y':y_collapsed, 'bed':B}

In [244]:
surfile = scipy.io.savemat('surface_idealized.mat', surface_dict)
bedfile = scipy.io.savemat('bed_idealized.mat', bed_dict)

# Another option: Valley glacier topography given by De Fleurian et al. (SHMIP) et al. 2018

In [19]:
# https://www.cambridge.org/core/journals/journal-of-glaciology/article/shmip-the-subglacial-hydrology-model-intercomparison-project/AA648F9E8B0056A4625531C731AE564D

### Bed

In [234]:
Lx = 6000
Ly = 1050

In [236]:
gamma_b = 0.05  # reference parameter

def zs(x, y):  # surface elevation — does not depend on y, but matches eq. 12 signature
    return 100 * (x + 200)**(1/4) + x/60 - (2e10)**(1/4) + 1

def f(x, gamma):  # helper 1
    return ((zs(6000, 0) - 6000 * gamma) / 6000**2) * x**2 + gamma * x

def g(y):  # helper 2
    return 0.5e-6 * np.abs(y)**3

def g_inv(val):
    # g(y) = 0.5e-6 * |y|^3  =>  y = (val / 0.5e-6)^(1/3)
    return (val / 0.5e-6)**(1/3)

def h(x, gamma):  # helper 3
    numerator   = (-4.5 * x / 6000 + 5) * (zs(x, 0) - f(x, gamma))
    denominator = zs(x, 0) - f(x, gamma_b) + 1e-16
    return numerator / denominator

def zb(x, y, gamma):  # bed elevation
    return f(x, gamma) + g(y) * h(x, gamma)

def y0(x):
    # eq. 14: y_o(x) = g^{-1}( (zb(x,0,gamma_b) - f(x,gamma_b)) / (h(x,gamma_b) + 1e-16) )
    numerator   = zb(x, 0, gamma_b) - f(x, gamma_b)
    denominator = h(x, gamma_b) + 1e-16
    return g_inv(numerator / denominator)

x = np.linspace(0, Lx, Lx)
y = np.linspace(-Ly/2, Ly/2, Ly)  # ← symmetric about y=0

X, Y = np.meshgrid(x, y)
gamma = 0

surf = zs(X, Y)
bed  = zb(X, Y, gamma)



In [237]:
plt.contourf(X, Y, surf)
#plt.contourf(X, Y, bed)

In [248]:
np.max(Y)

np.float64(525.0)

In [249]:
thick = surf - bed
fig, ax = plt.subplots()
cp = plt.contourf(X, Y, thick, 50)
plt.axis('scaled')
plt.plot(y0(x))

# 2. Add the colorbar, specifying which plot it describes
fig.colorbar(cp, ax=ax)

In [250]:
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(X, Y, surf, cmap = 'Blues', alpha=1, edgecolor='none', label='surf')
ax.plot_surface(X, Y, bed, cmap='copper', alpha = 0.5, edgecolor='none', label='bed')
ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.set_zlabel('z (m)')


Text(0.5, 0, 'z (m)')

In [251]:
x_collapsed = X[0, :]
y_collapsed = Y[:, 0]

surface_dict = {'X':x_collapsed, 'Y': y_collapsed,'surface':surf}
bed_dict = {'X':x_collapsed, 'Y':y_collapsed, 'bed':bed}



In [252]:
surfile = scipy.io.savemat('surface_idealized.mat', surface_dict)
bedfile = scipy.io.savemat('bed_idealized.mat', bed_dict)

In [133]:
surfile = scipy.io.savemat('surface_idealized_shmip.mat', surface_dict)
bedfile = scipy.io.savemat('bed_idealized_shmip.mat', bed_dict)

### Generate exp file

In [194]:
cont = plt.contourf(X, Y, surf-bed, 0)


In [195]:
cont.allsegs[0]

[array([[   1.00016669, -500.        ],
        [   2.00033339, -500.        ],
        [   3.00050008, -500.        ],
        ...,
        [   0.        , -498.998999  ],
        [   0.        , -500.        ],
        [   1.00016669, -500.        ]], shape=(5246, 2)),
 array([[5343.89064844, -500.        ],
        [5344.89081514, -500.        ],
        [5345.89098183, -500.        ],
        ...,
        [5343.89064844, -499.89575633],
        [5342.92906712, -500.        ],
        [5343.89064844, -500.        ]], shape=(4625, 2)),
 array([[ 0.08188642, 74.57457457],
        [ 0.17651395, 75.57557558],
        [ 0.27432794, 76.57657658],
        ...,
        [ 0.        , 74.57457457],
        [ 0.        , 73.6793403 ],
        [ 0.08188642, 74.57457457]], shape=(5246, 2))]

In [206]:
cont_segs = cont.allsegs[0][0]
cont_segs2 = cont.allsegs[0][1]
cont_segs3 = cont.allsegs[0][2]

cont_segs_combined = np.concatenate([cont_segs, cont_segs2, cont_segs3], 0)

plt.plot(cont_segs_combined[:, 0], cont_segs_combined[:, 1])
#plt.plot(cont_segs[:, 0], cont_segs[:, 1])
#plt.plot(cont_segs2[:, 0], cont_segs2[:, 1])
#plt.plot(cont_segs3[:, 0], cont_segs3[:, 1])
 

In [208]:
X_segs = cont_segs_combined[:, 0]
Y_segs = cont_segs_combined[:, 1]

In [210]:
# Downsample X_int and Y_int
X_downsamp = X_segs[::50]
Y_downsamp = Y_segs[::50]
len(X_downsamp)

303

In [211]:
np.savetxt('intersection_coords.txt', np.column_stack([X_downsamp, Y_downsamp]), 
           fmt='%.6f', header='Xpos Ypos', comments='')